## 1. Architecture

### Retrieval-Augmented Generation (RAG) for ICU Clinical Q&A

This notebook implements a **Retrieval-Augmented Generation (RAG)** pipeline that
allows clinical users to query patient risk information in natural language.
The architecture has two phases:

**INGESTION (offline, done once):**

```
ICU patient features + SHAP values
        |
        v
build_patient_documents()  -->  LangChain Document objects
        |                        (one per patient, with metadata)
        v
HuggingFace Embeddings (sentence-transformers/all-MiniLM-L6-v2)
        |
        v
Chroma Vector Store  (persisted to disk)
```

**QUERY (online, per user question):**

```
User question (natural language)
        |
        v
Vector similarity search  -->  Top-k relevant patient docs
        |
        v
Claude (Retrieval Q&A chain)  -->  Grounded answer
```

**Key design choices:**
- **HuggingFace embeddings** (local, no API call): avoids sending patient data
  to external embedding services, preserving privacy.
- **Chroma vector store**: lightweight, file-based persistence — no database server
  required for prototyping.
- **SHAP values in documents**: each patient document includes their top risk
  factors (by SHAP magnitude), enabling the LLM to explain predictions in
  natural language without hallucinating feature importances.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('../'))

import pandas as pd
import numpy as np
import json
from dotenv import load_dotenv

from src.models import load_model
from src.preprocessing import split_data, scale_numeric
from src.explainability import compute_shap_values
from src.langchain_qa import (
    build_patient_documents,
    build_vector_store,
    build_qa_chain,
    ask
)

# Load environment variables (ANTHROPIC_API_KEY, etc.)
load_dotenv(dotenv_path='../.env')

api_key = os.getenv('ANTHROPIC_API_KEY')
if not api_key:
    print('WARNING: ANTHROPIC_API_KEY not found in environment. Q&A chain will not work.')
    print('Set ANTHROPIC_API_KEY in the .env file at the project root.')
else:
    print('ANTHROPIC_API_KEY loaded successfully.')

# Load champion model
model = load_model('lgbm_best')
print(f'Model loaded: {type(model).__name__}')

# Load and prepare test data
df = pd.read_csv('../data/processed/features_engineered.csv')
X_train, X_val, X_test, y_train, y_val, y_test = split_data(df, target='hospital_death')
X_train_s, X_val_s, X_test_s, scaler = scale_numeric(X_train, X_val, X_test)

# Get predictions for the test set
y_pred_proba = model.predict_proba(X_test_s)[:, 1]

print(f'Test set size: {len(X_test):,}')
print(f'Prediction range: [{y_pred_proba.min():.4f}, {y_pred_proba.max():.4f}]')

## 2. Build Patient Documents

Each patient is converted to a LangChain `Document` object. The document's `page_content`
is a structured natural-language summary including:
- Patient identifier and predicted mortality probability.
- Top 5 risk factors with their SHAP values (positive = increases risk, negative = protective).
- Key demographic and clinical context.

The document `metadata` stores structured fields (patient ID, predicted probability,
actual outcome) for post-retrieval filtering.

We work on a **500-patient sample** to keep vector store building fast for this demo;
production deployment would index all test patients.

In [ ]:
# Sample 500 patients from the test set for the demo
np.random.seed(42)
n_sample = min(500, len(X_test_s))
sample_idx = np.random.choice(len(X_test_s), size=n_sample, replace=False)

if hasattr(X_test_s, 'iloc'):
    X_sample = X_test_s.iloc[sample_idx].reset_index(drop=True)
else:
    X_sample = X_test_s[sample_idx]

y_sample_proba = y_pred_proba[sample_idx]
y_sample_true = (
    np.array(y_test)[sample_idx]
    if hasattr(y_test, '__len__')
    else y_test.values[sample_idx]
)

print(f'Sample size: {n_sample}')
print(f'Computing SHAP values for {n_sample} patients...')

shap_values = compute_shap_values(model, X_sample)

# Build SHAP dict: patient_id -> {feature: shap_value}
feature_names = (
    X_sample.columns.tolist()
    if hasattr(X_sample, 'columns')
    else [f'feature_{i}' for i in range(X_sample.shape[1])]
)

shap_dict = {
    i: dict(zip(feature_names, shap_values.values[i]))
    for i in range(n_sample)
}

# Build patient documents
documents = build_patient_documents(
    X=X_sample,
    y_pred_proba=y_sample_proba,
    y_true=y_sample_true,
    shap_dict=shap_dict,
    feature_names=feature_names
)

print(f'\nBuilt {len(documents)} patient documents.')
print('\n--- Example Document (Patient 0) ---')
print(documents[0].page_content)
print('\nMetadata:')
print(json.dumps(documents[0].metadata, indent=2, default=str))

## 3. Build Vector Store

We embed all patient documents using the `all-MiniLM-L6-v2` model from
HuggingFace sentence-transformers — a 22M-parameter model that produces
high-quality 384-dimensional embeddings very quickly.

Documents are indexed in a **Chroma** vector store and persisted to
`../data/processed/chroma_db/` so the index does not need to be rebuilt
on every notebook run. The `persist=True` flag in `build_vector_store` handles
this automatically.

In [ ]:
persist_dir = '../data/processed/chroma_db'
os.makedirs(persist_dir, exist_ok=True)

print(f'Building vector store from {len(documents)} documents...')
print('This may take 30-60 seconds for 500 documents...')

vector_store = build_vector_store(
    documents=documents,
    persist=True,
    persist_directory=persist_dir
)

# Report index size
collection = vector_store._collection
n_indexed = collection.count()
print(f'\nVector store built successfully.')
print(f'Number of chunks indexed: {n_indexed}')
print(f'Persistence directory: {persist_dir}')

# Test retrieval with a simple query
test_query = 'high risk elderly patient with elevated heart rate'
retrieved = vector_store.similarity_search(test_query, k=3)
print(f'\nTest retrieval for: "{test_query}"')
print(f'Retrieved {len(retrieved)} documents. First result preview:')
print(retrieved[0].page_content[:300] + '...')

## 4. Build Q&A Chain and Demonstrate

The `build_qa_chain` function wires together the vector store retriever and
a Claude language model into a LangChain `RetrievalQA` chain. The chain:
1. Embeds the user question and retrieves the top-k most relevant patient documents.
2. Passes the retrieved context plus the question to Claude.
3. Returns a grounded answer citing specific patients and their SHAP-derived risk factors.

We demonstrate the interface with three clinically meaningful questions:
- A cohort-level question about highest-risk patients.
- A feature-level question about APACHE II score influence.
- A patient-level question targeting age as a dominant driver.

In [ ]:
qa_chain = build_qa_chain(vector_store=vector_store)
print(f'Q&A chain built: {type(qa_chain).__name__}')

questions = [
    'Which patients are at highest risk and what are their key risk factors?',
    'How does APACHE II score influence the predictions?',
    'Are there any patients where age is the dominant risk driver?'
]

print('\n' + '=' * 70)
print('ICU MORTALITY RISK -- CLINICAL Q&A DEMONSTRATION')
print('=' * 70)

answers = []
for i, question in enumerate(questions, 1):
    print(f'\n[Q{i}] {question}')
    print('-' * 60)
    try:
        answer = ask(qa_chain, question)
        answers.append({'question': question, 'answer': answer})
        print(f'[A{i}] {answer}')
    except Exception as e:
        err_msg = f'[Q&A chain error: {type(e).__name__}: {e}]'
        answers.append({'question': question, 'answer': err_msg})
        print(f'[A{i}] {err_msg}')

print('\n' + '=' * 70)

# Save Q&A log
qa_log_path = '../reports/qa_demo_log.json'
os.makedirs('../reports', exist_ok=True)
with open(qa_log_path, 'w') as f:
    json.dump(answers, f, indent=2)
print(f'\nQ&A log saved to {qa_log_path}')

## 5. Closing Notes

### Limitations of RAG over Tabular Data

While the RAG Q&A interface provides an intuitive natural-language window into
patient risk data, several limitations must be acknowledged before clinical use:

**1. Embedding space does not equal clinical similarity.** Sentence transformers
were designed for text similarity, not tabular medical data. Two patients may be
clinically similar but have very different document representations, or text-similar
but clinically dissimilar. SHAP-enriched documents help, but do not fully bridge
this gap.

**2. Faithfulness vs. accuracy.** The LLM answers are grounded in retrieved
documents (reducing hallucination), but Claude may still mis-summarise numerical
information or conflate details from different patients in the context window.
Answers should be verified against the raw `documents` objects.

**3. Privacy.** This prototype operates on test-set data that is not de-identified
beyond the scope of the WiDS dataset. Production deployment requires formal
de-identification, access controls, and compliance with HIPAA / GDPR.

**4. Top-k retrieval is not a global ranking.** Vector similarity search retrieves
the most text-similar documents, not necessarily the highest-risk patients. For
precise top-N queries, combine with structured metadata filtering
(e.g., `predicted_prob >= 0.7`) using Chroma's `where` clause.

### Next Steps

- **Hybrid retrieval**: combine dense vector search with BM25 keyword search
  (Elasticsearch or LangChain's `EnsembleRetriever`) for better coverage.
- **Structured tool use**: expose the dataframe as a callable tool so Claude can
  execute precise pandas queries in addition to semantic search.
- **Streaming interface**: wrap the chain in a Gradio or Streamlit app for
  interactive clinical demos with real-time streaming responses.
- **Re-ranking**: use a cross-encoder reranker on retrieved documents before
  passing to Claude to improve answer grounding on complex questions.